# 02 · Segmentation and density counting (drill)

**Style B: blind-code drill.** Write each piece from memory, run the check, delete your code.
Reference: `../solutions/drills/02_segmentation_density.py`.

**Traces to** `notes/ioai-task-patterns.md` P8 · T25-RADAR (label shift, weighted pixel metric), T25-H-WEATHER
(Dice), T25-CHICKEN (non-negative density maps whose sum is the count). Convolutional layers are a syllabus
"Both" item, so the U-Net is written by hand.

Total suggested time: **55 min**.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

def make_maps(n, seed, H=32, W=64):
    rng = np.random.default_rng(seed); yy, xx = np.mgrid[:H, :W]
    X = rng.gamma(2.0, 0.1, (n, 3, H, W)).astype(np.float32); Y = -np.ones((n, H, W), np.int64)
    sig = np.array([[1.0, 0.2, 0.6], [0.3, 1.0, 0.5]])
    for i in range(n):
        for _ in range(rng.integers(1, 4)):
            c = rng.integers(0, 2); cy, cx = rng.uniform(3, H - 3), rng.uniform(3, W - 3)
            blob = ((yy - cy) / rng.uniform(1.5, 3)) ** 2 + ((xx - cx) / rng.uniform(1.5, 4)) ** 2 <= 1
            X[i][:, blob] += (rng.uniform(0.4, 1.0) * sig[c])[:, None]; Y[i][blob] = c
    return X, Y
X_tr, Y_tr = make_maps(1200, 0); X_va, Y_va = make_maps(300, 1)

## 1. Dataset ⏱ 4 min

`MapDataset(X, Y)`: a `torch.utils.data.Dataset` whose item `i` is `(x float32 (C,H,W), y int64 (H,W))` with the
labels shifted from −1..1 to 0..2.

In [ ]:
d = MapDataset(X_tr, Y_tr)
x, y = d[5]
assert len(d) == 1200 and x.dtype == torch.float32 and x.shape == (3, 32, 64)
assert y.dtype == torch.int64 and y.shape == (32, 64) and torch.equal(y, torch.from_numpy(Y_tr[5]) + 1)
print("section 1 ok")

## 2. U-Net ⏱ 12 min

`UNet(in_ch, n_classes)`: encoder–decoder with at least 2 down/up levels and skip connections. It maps
`(B, in_ch, H, W)` to logits `(B, n_classes, H, W)` for H, W divisible by 4, with < 2M parameters.

In [ ]:
u = UNet(3, 3)
assert u(torch.randn(2, 3, 32, 64)).shape == (2, 3, 32, 64) and u(torch.randn(1, 3, 48, 96)).shape == (1, 3, 48, 96)
assert sum(p.numel() for p in u.parameters()) < 2_000_000
print("section 2 ok")

## 3. Weighted pixel score and Dice ⏱ 7 min

- `radar_score(pred, target, bonus=50) -> float`: T25-RADAR's metric on **original** labels (−1 = background):
  correct background pixels count 1, correct object pixels count `bonus`, normalised by the maximum possible.
- `dice(pred_mask, true_mask, eps=1e-6) -> float`: `(2|P∩T| + eps) / (|P| + |T| + eps)` for boolean masks.

In [ ]:
gt = torch.tensor([[-1, -1, -1], [1, 2, 3], [-1, -1, -1]]); pr = torch.tensor([[-1, 1, -1], [-1, 2, -1], [-1, 3, -1]])
assert abs(radar_score(pr, gt) - 54 / 156) < 1e-6
assert abs(dice(torch.tensor([1, 1, 0, 0]), torch.tensor([1, 0, 1, 0])) - 0.5) < 1e-4 and abs(dice(torch.zeros(4), torch.zeros(4)) - 1) < 1e-4
print("section 3 ok")

## 4. Train the segmenter ⏱ 12 min

Train a `UNet(3, 3)` called `seg` on `MapDataset(X_tr, Y_tr)`. Hint: use a weighted loss that mirrors the metric.
The check needs `radar_score ≥ 0.85` on the validation maps.

In [ ]:
seg.eval()
with torch.no_grad():
    pred = torch.cat([seg(torch.from_numpy(X_va[i:i + 100]).to(device)).argmax(1).cpu() for i in range(0, 300, 100)]) - 1
s = radar_score(pred, torch.from_numpy(Y_va))
assert s >= 0.85, s
print(f"section 4 ok (radar score {s:.4f}, object Dice {dice(pred >= 0, torch.from_numpy(Y_va) >= 0):.3f})")

## 5. Counting by density ⏱ 14 min

`count_images(imgs) -> Tensor (n,)`: predicted counts for `(n, 1, 64, 96)` images, from a model you train on
`img_tr, den_tr` (full-resolution density maps whose sum is the count). The model must output a **non-negative**
density map. The check computes T25-CHICKEN's `exp(-mean relative error)` and needs ≥ 0.85.

In [ ]:
def make_counts(n, seed, H=64, W=96, s=1.5):
    rng = np.random.default_rng(seed); yy, xx = np.mgrid[:H, :W]
    img = rng.normal(0.2, 0.05, (n, 1, H, W)).astype(np.float32); den = np.zeros((n, 1, H, W), np.float32)
    for i in range(n):
        for _ in range(rng.poisson(12)):
            cy, cx = rng.uniform(2, H - 2), rng.uniform(2, W - 2)
            g = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * s * s)); img[i, 0] += 0.8 * g; den[i, 0] += g / g.sum()
    return torch.from_numpy(img), torch.from_numpy(den)
img_tr, den_tr = make_counts(800, 2); img_va, den_va = make_counts(200, 3)

In [ ]:
pc = torch.as_tensor(count_images(img_va)).float().flatten(); tc = den_va.sum((1, 2, 3))
score = float(np.exp(-((pc - tc).abs() / tc).mean()))
assert pc.shape == (200,) and score >= 0.85, score
print(f"section 5 ok (T25-CHICKEN score {score:.3f})")